# 04 · Evaluation (test set, once)
> **Research prototype.** Predicts *dataset-defined* dementia categories from preprocessed JPG brain MRI slices. Not a clinically validated diagnostic system.

Steps 20–25 and 30–32. `best_model.pth` of each model (selected on validation macro-F1) is evaluated **once** on the
untouched test set; a lock file stops a different checkpoint from being evaluated on the test set later. Temperature
scaling is fitted on validation only. Then: binary non-demented vs any-dementia (derived from the four-class
probabilities), ordinal severity analysis, calibration, uncertainty (entropy + MC dropout), robustness on transformed
copies, ablation (validation only) and external validation if configured.

In [ ]:
# Bootstrap: locate the project code (local checkout, /kaggle/working, or a Kaggle dataset upload)
import os, sys, glob, shutil, json
from pathlib import Path

CANDIDATES = [Path.cwd().parent, Path.cwd(), Path("/kaggle/working/dementia_mri_project")]
CANDIDATES += [Path(p).parent for p in glob.glob("/kaggle/input/*/dementia_mri_project/src")
               + glob.glob("/kaggle/input/*/*/dementia_mri_project/src") + glob.glob("/kaggle/input/*/src")]
CODE = next((c for c in CANDIDATES if (c / "src" / "run_pipeline.py").exists()), None)
if CODE is None:
    raise FileNotFoundError("Project code not found. On Kaggle: upload dementia_mri_project.zip as a Dataset and add it "
                            "as an Input (Kaggle extracts it), or put the project folder next to this notebook.")
if str(CODE).startswith("/kaggle/input"):                       # /kaggle/input is read-only -> work in /kaggle/working
    dst = Path("/kaggle/working/dementia_mri_project")
    shutil.copytree(CODE / "src", dst / "src", dirs_exist_ok=True)
    shutil.copy(CODE / "predict.py", dst / "predict.py")
    if not (dst / "configs" / "config.json").exists():
        shutil.copytree(CODE / "configs", dst / "configs", dirs_exist_ok=True)
    for sub in ("data", "checkpoints", "results", "gradcam", "reports"):   # outputs of earlier notebooks, if attached
        if (CODE / sub).exists() and not (dst / sub).exists():
            shutil.copytree(CODE / sub, dst / sub)
    CODE = dst
sys.path.insert(0, str(CODE)); os.chdir(CODE)
print("Project code:", CODE)

from src.utils import load_config, get_paths, print_env, load_json, save_json
from src.nb_helpers import show, show_md, show_table, sample_grid

CONFIG_OVERRIDES = {}      # e.g. {"batch_size": 16, "num_workers": 4, "epochs_stage2": 30}
cfg = load_config(overrides=CONFIG_OVERRIDES)
paths = get_paths(cfg)
ENV = print_env()
print("Outputs ->", paths["root"])

In [ ]:
from src.run_pipeline import stage_evaluate
RES = stage_evaluate(cfg, paths, robustness=True, ablation=cfg["ablation"]["enabled"])

## Four-class results

In [ ]:
import pandas as pd
show_table(paths["results"] / "classification_report.csv")
show(paths["results"] / "confusion_matrix.png", 520); show(paths["results"] / "normalized_confusion_matrix.png", 520)
show(paths["results"] / "roc_curves.png", 520); show(paths["results"] / "pr_curves.png", 520)
M = load_json(paths["results"] / "metrics.json")
print(json.dumps(M["class_specific"], indent=1))

## Binary dementia detection (derived) and ordinal severity analysis

In [ ]:
print(json.dumps(M["binary_dementia_detection"], indent=1)); show(paths["results"] / "binary_dementia_curves.png", 700)
print(json.dumps(M["ordinal_severity"], indent=1))

## Calibration and uncertainty

In [ ]:
c = M["calibration"]
print(f"T = {c['temperature']:.3f} | ECE {c['uncalibrated']['ece']:.4f} -> {c['temperature_scaled']['ece']:.4f} | "
      f"Brier {c['uncalibrated']['brier']:.4f} -> {c['temperature_scaled']['brier']:.4f} | "
      f"accuracy when confidence >= 0.9: {c['accuracy_when_confidence_ge_0.9']}")
show(paths["results"] / "calibration.png", 800); show(paths["results"] / "uncertainty.png", 900)
print(json.dumps(M["uncertainty"], indent=1, default=str))

## Baseline vs primary, robustness, ablation, external validation, model size

In [ ]:
display(pd.read_csv(paths["results"] / "model_comparison.csv").round(4))
display(pd.read_csv(paths["results"] / "robustness_results.csv").round(4))
if (paths["results"] / "ablation_results.csv").exists():
    display(pd.read_csv(paths["results"] / "ablation_results.csv").round(4))
print(load_json(paths["results"] / "external_validation.json"))
print(json.dumps(M["performance"], indent=1))